# Interpolação e Extrapolação Populacional — Método Geométrico
### Todos os Municípios de MG (1970-2025), por Faixa Etária


In [1]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# utilitarios_comuns.py mora na mesma pasta deste notebook -- Path.cwd()
# funciona porque o Jupyter abre o kernel já nessa pasta.
sys.path.append(str(Path.cwd()))

from utilitarios_comuns import (
    COLUNA_MUNICIPIO,
    COLUNA_CODIGO_IBGE,
    COLUNA_REGIONAL,
    POPULACAO_MINIMA_VALIDA,
    SUFIXO_COLUNA_TOTAL,
    PASTA_POPULACAO,
    CAMINHO_POPULACAO,
)

### Configurações

In [2]:
CAMINHO_ARQUIVO_ENTRADA = PASTA_POPULACAO / "tabela_populacao_faixa_etaria.csv"
# Mesmo arquivo que utilitarios_comuns.CAMINHO_POPULACAO -- fonte única
# do caminho de saída, já que calculo_taxa.ipynb e os notebooks de
# gráfico também dependem dele.
CAMINHO_ARQUIVO_SAIDA = CAMINHO_POPULACAO

LISTA_FAIXAS_ETARIAS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70 OU MAIS"]

LISTA_ANOS_CENSITARIOS = [1970, 1980, 1991, 2000, 2010, 2022]
ANO_INICIO = 1970
ANO_FIM = 2025
LISTA_ANOS_COMPLETOS = list(range(ANO_INICIO, ANO_FIM + 1))

### Leitura e Validação das Colunas

Confere se todas as colunas esperadas existem no arquivo de entrada:
uma coluna `<ano>_<faixa>` para cada combinação de ano censitário e
faixa etária, e uma coluna `<ano>_TOTAL` para cada ano censitário
(usada só como checagem/diagnóstico, não como fonte da interpolação).

In [3]:
tabela_populacao = pd.read_csv(CAMINHO_ARQUIVO_ENTRADA)

colunas_faixa_esperadas = []
for ano in LISTA_ANOS_CENSITARIOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        colunas_faixa_esperadas.append(f"{ano}_{faixa}")

colunas_total_esperadas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_CENSITARIOS]

colunas_faltantes = [c for c in colunas_faixa_esperadas + colunas_total_esperadas if c not in tabela_populacao.columns]
if colunas_faltantes:
    raise ValueError(f"Colunas esperadas e não encontradas no arquivo de entrada: {colunas_faltantes}")

print(f"Arquivo de entrada validado: {tabela_populacao.shape[0]} municípios.")
print(f"Colunas de faixa etária esperadas ({len(colunas_faixa_esperadas)}): OK")
print(f"Colunas de total esperadas ({len(colunas_total_esperadas)}): OK")

tabela_populacao.head()

Arquivo de entrada validado: 853 municípios.
Colunas de faixa etária esperadas (48): OK
Colunas de total esperadas (6): OK


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2022_40-49,2022_50-59,2022_60-69,2022_70 OU MAIS,1970_TOTAL,1980_TOTAL,1991_TOTAL,2000_TOTAL,2010_TOTAL,2022_TOTAL
0,abadia dos dourados,3100104,uberlandia,1542,1255,791,537,274,207,143,...,493,465,377,339,4848,3936,3252,3105,3676,3111
1,abaete,3100203,sete lagoas,3054,2786,1423,1115,874,470,311,...,1543,1752,1436,1250,10206,9926,10560,10947,11171,11474
2,abre campo,3100302,manhumirim,3015,2534,1462,1049,678,543,311,...,960,903,748,706,9779,9375,9131,6610,7238,6903
3,acaiaca,3100401,ponte nova,514,387,200,208,135,128,70,...,304,290,222,235,1699,1822,1937,1875,1881,2012
4,aguanil,3100807,divinopolis,359,302,217,187,126,75,23,...,298,303,260,215,1312,1560,1576,1753,1938,2196


### Preparar a tabela indexada por Código IBGE

Todas as etapas seguintes (interpolação, checagens) trabalham sobre
essa tabela, indexada pelo Código IBGE — garante alinhamento correto
entre município e valor, independente da ordem das linhas.

In [4]:
tabela_indexada = tabela_populacao.set_index(COLUNA_CODIGO_IBGE, drop=False)

tabela_indexada[colunas_faixa_esperadas] = tabela_indexada[colunas_faixa_esperadas].where(
    tabela_indexada[colunas_faixa_esperadas] >= POPULACAO_MINIMA_VALIDA
)

tabela_indexada.head()

,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2022_40-49,2022_50-59,2022_60-69,2022_70 OU MAIS,1970_TOTAL,1980_TOTAL,1991_TOTAL,2000_TOTAL,2010_TOTAL,2022_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,uberlandia,1542.0,1255.0,791.0,537.0,274.0,207.0,143.0,...,493,465,377,339,4848,3936,3252,3105,3676,3111
3100203,abaete,3100203,sete lagoas,3054.0,2786.0,1423.0,1115.0,874.0,470.0,311.0,...,1543,1752,1436,1250,10206,9926,10560,10947,11171,11474
3100302,abre campo,3100302,manhumirim,3015.0,2534.0,1462.0,1049.0,678.0,543.0,311.0,...,960,903,748,706,9779,9375,9131,6610,7238,6903
3100401,acaiaca,3100401,ponte nova,514.0,387.0,200.0,208.0,135.0,128.0,70.0,...,304,290,222,235,1699,1822,1937,1875,1881,2012
3100807,aguanil,3100807,divinopolis,359.0,302.0,217.0,187.0,126.0,75.0,23.0,...,298,303,260,215,1312,1560,1576,1753,1938,2196


### Checagem de Valores Não Positivos, por Ano Censitário e Faixa Etária

Mostra, para cada combinação de ano e faixa, quantos municípios ficaram
sem dado válido -- útil para confirmar se o padrão bate com o esperado
(municípios ainda não emancipados nos censos mais antigos).

In [5]:
tabela_contagem_ausentes = pd.DataFrame(index=LISTA_ANOS_CENSITARIOS, columns=LISTA_FAIXAS_ETARIAS, dtype=int)

for ano in LISTA_ANOS_CENSITARIOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        coluna = f"{ano}_{faixa}"
        tabela_contagem_ausentes.loc[ano, faixa] = int(tabela_indexada[coluna].isna().sum())

print("Municípios sem dado válido (ausente/<= 0), por ano censitário x faixa etária:")
tabela_contagem_ausentes

Municípios sem dado válido (ausente/<= 0), por ano censitário x faixa etária:


,0-9,10-19,20-29,30-39,40-49,50-59,60-69,70 OU MAIS
1970,131.0,131.0,131.0,131.0,131.0,131.0,131.0,133.0
1980,131.0,131.0,131.0,130.0,131.0,131.0,131.0,132.0
1991,130.0,130.0,130.0,0.0,130.0,130.0,130.0,130.0
2000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2010,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Método Geométrico

Funções com as fórmulas do método geométrico discreto — as mesmas usadas
antes, agora aplicadas separadamente a cada faixa etária.

In [6]:
def calcular_taxa_geometrica(populacao_inicial, populacao_final, tempo_total):
    populacao_inicial_valida = populacao_inicial.where(populacao_inicial >= POPULACAO_MINIMA_VALIDA)
    razao = populacao_final / populacao_inicial_valida
    return razao ** (1 / tempo_total) - 1


def estimar_populacao_geometrica(populacao_base, taxa_geometrica, tempo_decorrido):
    return populacao_base * (1 + taxa_geometrica) ** tempo_decorrido

### Interpolação e Extrapolação

In [7]:
def interpolar_geometrica(tabela, colunas_censitarias, anos_censitarios, anos_completos):
    P = tabela[colunas_censitarias].copy()
    P.columns = anos_censitarios

    resultado = pd.DataFrame(index=P.index, columns=anos_completos, dtype=float)

    for ano in anos_censitarios:
        resultado[ano] = P[ano]

    for i in range(len(anos_censitarios) - 1):
        ano_inicial = anos_censitarios[i]
        ano_final = anos_censitarios[i + 1]

        populacao_inicial = P[ano_inicial]
        populacao_final = P[ano_final]
        tempo_total = ano_final - ano_inicial

        r_g = calcular_taxa_geometrica(populacao_inicial, populacao_final, tempo_total)

        anos_intermediarios = [ano for ano in anos_completos if ano_inicial < ano < ano_final]
        for ano in anos_intermediarios:
            tempo_decorrido = ano - ano_inicial
            resultado[ano] = estimar_populacao_geometrica(populacao_inicial, r_g, tempo_decorrido)

    return resultado


def extrapolar_geometrica(tabela_interpolada, ano_inicial, ano_final, anos_a_extrapolar):
    P_inicial = tabela_interpolada[ano_inicial]
    P_final = tabela_interpolada[ano_final]
    t = ano_final - ano_inicial

    r_g = calcular_taxa_geometrica(P_inicial, P_final, t)

    extrapolacao = pd.DataFrame(index=tabela_interpolada.index)
    for ano in anos_a_extrapolar:
        extrapolacao[ano] = estimar_populacao_geometrica(P_final, r_g, ano - ano_final)

    return extrapolacao, r_g


ULTIMOS_DOIS_CENSOS = LISTA_ANOS_CENSITARIOS[-2:]  # [2010, 2022]
ultimo_ano_censitario = LISTA_ANOS_CENSITARIOS[-1]

ANOS_A_EXTRAPOLAR = [ano for ano in LISTA_ANOS_COMPLETOS if ano > ultimo_ano_censitario]

print("Últimos dois censos usados na taxa de extrapolação:", ULTIMOS_DOIS_CENSOS)
print("Anos extrapolados                                 :", ANOS_A_EXTRAPOLAR)

Últimos dois censos usados na taxa de extrapolação: [2010, 2022]
Anos extrapolados                                 : [2023, 2024, 2025]


### Interpolar e extrapolar cada faixa etária, separadamente

Para cada faixa, roda a mesma interpolação/extrapolação já usada antes
para o total — só que agora aplicada às colunas daquela faixa
especificamente. O resultado é um dicionário com uma tabela completa
(1970-2025) por faixa etária.

In [8]:
tabelas_por_faixa = {}

for faixa in LISTA_FAIXAS_ETARIAS:
    colunas_censitarias_da_faixa = [f"{ano}_{faixa}" for ano in LISTA_ANOS_CENSITARIOS]

    tabela_interpolada = interpolar_geometrica(
        tabela_indexada, colunas_censitarias_da_faixa, LISTA_ANOS_CENSITARIOS, LISTA_ANOS_COMPLETOS
    )

    tabela_extrapolada, _ = extrapolar_geometrica(
        tabela_interpolada,
        ano_inicial=ULTIMOS_DOIS_CENSOS[0],
        ano_final=ULTIMOS_DOIS_CENSOS[1],
        anos_a_extrapolar=ANOS_A_EXTRAPOLAR,
    )

    tabela_completa_da_faixa = tabela_interpolada.copy()
    for ano in tabela_extrapolada.columns:
        tabela_completa_da_faixa[ano] = tabela_completa_da_faixa[ano].fillna(tabela_extrapolada[ano])

    # Arredonda para inteiro (contagem de pessoas), sempre que possível,
    # antes de somar as faixas -- evita acumular erro de arredondamento
    # entre as 8 somas parciais.
    tabela_completa_da_faixa = (
        np.round(tabela_completa_da_faixa)
        .replace([np.inf, -np.inf], np.nan)
        .astype("Int64")
    )

    tabelas_por_faixa[faixa] = tabela_completa_da_faixa

print(f"Faixas etárias processadas: {list(tabelas_por_faixa.keys())}")
tabelas_por_faixa[LISTA_FAIXAS_ETARIAS[0]].head()

Faixas etárias processadas: ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70 OU MAIS']


,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,1542,1458,1379,1304,1233,1166,1103,1043,986,933,...,360,355,349,343,338,332,327,322,317,312
3100203,3054,2920,2792,2670,2553,2442,2335,2233,2135,2041,...,1288,1267,1247,1227,1207,1188,1169,1150,1132,1114
3100302,3015,2939,2864,2791,2721,2652,2584,2519,2455,2393,...,869,854,840,826,813,799,786,773,760,748
3100401,514,510,505,501,496,492,488,484,479,475,...,233,228,223,219,215,210,206,202,198,194
3100807,359,360,362,363,365,366,367,369,370,372,...,264,265,265,266,267,267,268,269,269,270


### Somar as faixas etárias para obter a população total de cada ano

A população total do ano deixa de ser interpolada diretamente -- passa
a ser a soma das 8 faixas já interpoladas/extrapoladas, para cada
município e cada ano.

In [9]:
tabela_populacao_total = pd.DataFrame(index=tabela_indexada.index, columns=LISTA_ANOS_COMPLETOS)

for ano in LISTA_ANOS_COMPLETOS:
    soma_das_faixas = None
    for faixa in LISTA_FAIXAS_ETARIAS:
        valor_da_faixa = tabelas_por_faixa[faixa][ano]
        soma_das_faixas = valor_da_faixa if soma_das_faixas is None else soma_das_faixas.add(valor_da_faixa, fill_value=0)
    tabela_populacao_total[ano] = soma_das_faixas

tabela_populacao_total = tabela_populacao_total.astype("Int64")

novos_nomes_das_colunas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_COMPLETOS]
tabela_populacao_total.columns = novos_nomes_das_colunas

tabela_populacao_total.head()

,1970_TOTAL,1971_TOTAL,1972_TOTAL,1973_TOTAL,1974_TOTAL,1975_TOTAL,1976_TOTAL,1977_TOTAL,1978_TOTAL,1979_TOTAL,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,4848,4735,4624,4523,4424,4332,4242,4159,4083,4007,...,3277,3233,3195,3164,3142,3123,3111,3105,3106,3115
3100203,10206,10128,10062,10007,9961,9929,9906,9894,9894,9903,...,11149,11177,11215,11263,11322,11392,11474,11569,11677,11799
3100302,9779,9720,9664,9613,9568,9524,9484,9451,9420,9397,...,6956,6933,6914,6901,6895,6895,6903,6917,6936,6963
3100401,1699,1710,1719,1730,1742,1753,1767,1780,1793,1806,...,1918,1930,1942,1955,1974,1992,2012,2035,2057,2084
3100807,1312,1332,1353,1374,1396,1420,1446,1473,1500,1528,...,2042,2065,2087,2112,2139,2167,2196,2228,2261,2297


### Total recalculado bate com o total original nos anos censitários?



In [10]:
divergencias_encontradas = 0

for ano in LISTA_ANOS_CENSITARIOS:
    coluna_total_original = f"{ano}{SUFIXO_COLUNA_TOTAL}"
    coluna_total_recalculada = f"{ano}{SUFIXO_COLUNA_TOTAL}"

    total_original = tabela_indexada[coluna_total_original]
    total_recalculado = tabela_populacao_total[coluna_total_recalculada]

    diferenca_absoluta = (total_original - total_recalculado).abs()
    quantidade_com_diferenca_relevante = int((diferenca_absoluta > 1).sum())  # tolerância de 1 hab. por arredondamento

    if quantidade_com_diferenca_relevante > 0:
        divergencias_encontradas += quantidade_com_diferenca_relevante
        print(
            f"ATENÇÃO ({ano}): {quantidade_com_diferenca_relevante} município(s) com diferença "
            f"maior que 1 habitante entre o total original e a soma das faixas."
        )

if divergencias_encontradas == 0:
    print("Checagem OK: em todos os anos censitários, a soma das faixas bate com o total original (± 1 hab.).")

Checagem OK: em todos os anos censitários, a soma das faixas bate com o total original (± 1 hab.).


## Exportação

Combina o resultado em uma única tabela por município, com **todas as
8 faixas etárias interpoladas/extrapoladas para cada ano de 1970 a
2025**, mais a população total (soma das faixas) de cada ano,
preservando `REGIONAL DE SAÚDE` vinda do arquivo de entrada. O padrão
de nomeação das colunas segue o mesmo do arquivo de entrada: todas as
colunas `<ano>_<faixa>` primeiro, depois o bloco de `<ano>_TOTAL`.

In [11]:

blocos_por_faixa = {}
for ano in LISTA_ANOS_COMPLETOS:
    for faixa in LISTA_FAIXAS_ETARIAS:
        blocos_por_faixa[f"{ano}_{faixa}"] = tabelas_por_faixa[faixa][ano]

tabela_faixas_completa = pd.DataFrame(blocos_por_faixa, index=tabela_indexada.index).astype("Int64")

colunas_faixa_ordenadas = [f"{ano}_{faixa}" for ano in LISTA_ANOS_COMPLETOS for faixa in LISTA_FAIXAS_ETARIAS]
colunas_total_ordenadas = [f"{ano}{SUFIXO_COLUNA_TOTAL}" for ano in LISTA_ANOS_COMPLETOS]

tabela_faixas_completa = tabela_faixas_completa[colunas_faixa_ordenadas]
tabela_populacao_total = tabela_populacao_total[colunas_total_ordenadas]

tabela_saida = tabela_indexada[[COLUNA_MUNICIPIO, COLUNA_CODIGO_IBGE, COLUNA_REGIONAL]].merge(
    tabela_faixas_completa, left_index=True, right_index=True
).merge(
    tabela_populacao_total, left_index=True, right_index=True
)

CAMINHO_ARQUIVO_SAIDA.parent.mkdir(parents=True, exist_ok=True)
tabela_saida.to_csv(CAMINHO_ARQUIVO_SAIDA, index=False)

print("Arquivo salvo em:", CAMINHO_ARQUIVO_SAIDA)
print("Shape:", tabela_saida.shape)
print(f"Período coberto: {ANO_INICIO}-{ANO_FIM} ({len(LISTA_ANOS_COMPLETOS)} anos)")
print(f"Colunas de faixa etária: {len(colunas_faixa_ordenadas)} | Colunas de total: {len(colunas_total_ordenadas)}")
tabela_saida.head()

Arquivo salvo em: /home/anna/tcc/Seleção e Transformação/População/populacao_interpolacao_extrapolacao.csv
Shape: (853, 507)
Período coberto: 1970-2025 (56 anos)
Colunas de faixa etária: 448 | Colunas de total: 56


,MUNICÍPIO,CÓDIGO IBGE,REGIONAL DE SAÚDE,1970_0-9,1970_10-19,1970_20-29,1970_30-39,1970_40-49,1970_50-59,1970_60-69,...,2016_TOTAL,2017_TOTAL,2018_TOTAL,2019_TOTAL,2020_TOTAL,2021_TOTAL,2022_TOTAL,2023_TOTAL,2024_TOTAL,2025_TOTAL
CÓDIGO IBGE,,,,,,,,,,,,,,,,,,,,,
3100104,abadia dos dourados,3100104,uberlandia,1542,1255,791,537,274,207,143,...,3277,3233,3195,3164,3142,3123,3111,3105,3106,3115
3100203,abaete,3100203,sete lagoas,3054,2786,1423,1115,874,470,311,...,11149,11177,11215,11263,11322,11392,11474,11569,11677,11799
3100302,abre campo,3100302,manhumirim,3015,2534,1462,1049,678,543,311,...,6956,6933,6914,6901,6895,6895,6903,6917,6936,6963
3100401,acaiaca,3100401,ponte nova,514,387,200,208,135,128,70,...,1918,1930,1942,1955,1974,1992,2012,2035,2057,2084
3100807,aguanil,3100807,divinopolis,359,302,217,187,126,75,23,...,2042,2065,2087,2112,2139,2167,2196,2228,2261,2297
